# CanvasXpress Notebook Renderer — Demo

This notebook demonstrates the **CanvasXpress Notebook Renderer** VS Code extension.

Each cell below emits the mime type `application/canvasxpress+json`. When the extension is
installed, VS Code draws that output as an interactive CanvasXpress chart. The payload is a
`{data, config}` object (optionally `events`, `width`, `height`) — the arguments to
`new CanvasXpress({...})`.

> Run each cell with a Python (Jupyter) kernel. If you open this file in plain Jupyter, the
> stored outputs will show as raw JSON — the interactive chart only appears in VS Code with the
> renderer installed.

## A tiny helper

`show()` publishes a CanvasXpress spec under the renderer's mime type.

In [ ]:
from IPython.display import display

def show(data, config, **kw):
    """Emit a CanvasXpress spec for the VS Code notebook renderer."""
    spec = {"data": data, "config": config}
    spec.update(kw)  # optional: events, width, height
    display({"application/canvasxpress+json": spec}, raw=True)

## 1. Bar chart

In [ ]:
show(
    data={"y": {
        "vars": ["Revenue"],
        "smps": ["Q1", "Q2", "Q3", "Q4"],
        "data": [[10, 14, 9, 17]],
    }},
    config={
        "graphType": "Bar",
        "title": "Quarterly Revenue",
        "xAxis": ["Revenue"],
        "smpTitle": "Quarter",
        "showLegend": False,
    },
)

## 2. Scatter plot

In [ ]:
show(
    data={"y": {
        # Scatter2D: smps are the axis dimensions, vars are the point ids
        # (one row of data per point, columns matching smps).
        "smps": ["Weight", "MPG"],
        "vars": ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8"],
        "data": [
            [3.2, 24], [2.8, 30], [4.1, 18], [3.6, 21],
            [2.4, 33], [3.9, 19], [2.1, 36], [4.5, 15],
        ],
    }},
    config={
        "graphType": "Scatter2D",
        "title": "Weight vs. MPG",
        "xAxis": ["Weight"],
        "yAxis": ["MPG"],
    },
)

## 3. Heatmap

A slightly larger dataset, and a custom `height`.

In [ ]:
show(
    data={"y": {
        "vars": ["Gene A", "Gene B", "Gene C", "Gene D"],
        "smps": ["S1", "S2", "S3", "S4", "S5", "S6"],
        "data": [
            [1.2, -0.4, 0.8, 1.9, -1.1, 0.3],
            [-0.9, 1.4, 0.2, -0.7, 1.8, -0.5],
            [0.5, 0.6, -1.3, 1.1, -0.2, 1.6],
            [1.7, -1.2, 0.9, -0.6, 0.4, -1.4],
        ],
    }},
    config={
        "graphType": "Heatmap",
        "title": "Expression Heatmap",
        "samplesClustered": True,
        "variablesClustered": True,
    },
    height=360,
)

## 4. Straight from a `DataFrame`

Usually your data is already a pandas `DataFrame`, not a hand-written `y` object. `show_df()`
converts one for you: **the index becomes `vars`, the columns become `smps`**, and the values
become the data matrix. That layout is exactly what a Heatmap (and most CanvasXpress graph
types) expect — features in rows, samples in columns.

In [ ]:
import pandas as pd

def show_df(df, config, **kw):
    """Render a pandas DataFrame: index -> vars, columns -> smps, values -> data."""
    data = {"y": {
        "vars": [str(v) for v in df.index],
        "smps": [str(s) for s in df.columns],
        "data": df.to_numpy().tolist(),
    }}
    show(data, config, **kw)


# A plain DataFrame: genes in rows, samples in columns.
expr = pd.DataFrame(
    [
        [1.2, -0.4, 0.8, 1.9, -1.1, 0.3],
        [-0.9, 1.4, 0.2, -0.7, 1.8, -0.5],
        [0.5, 0.6, -1.3, 1.1, -0.2, 1.6],
        [1.7, -1.2, 0.9, -0.6, 0.4, -1.4],
    ],
    index=["Gene A", "Gene B", "Gene C", "Gene D"],
    columns=["S1", "S2", "S3", "S4", "S5", "S6"],
)

show_df(expr, {
    "graphType": "Heatmap",
    "title": "Expression Heatmap (from a DataFrame)",
    "samplesClustered": True,
    "variablesClustered": True,
}, height=360)